# Twitter Airline Sentiment Analysis

**Goal:** Classify tweets as Positive, Neutral, Negative
**Algorithm:** Logistic Regression + TF-IDF
**Dataset:** [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment)

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Load Data

In [1]:
path = kagglehub.dataset_download('crowdflower/twitter-airline-sentiment')
df = pd.read_csv(f'{path}/Tweets.csv')
print('Shape:', df.shape)
print('Sentiment:', df['airline_sentiment'].value_counts().to_dict())

Shape: (14640, 15)
Sentiment: {'negative': 9178, 'neutral': 3099, 'positive': 2363}


## 2. Preprocess

In [1]:
import re
def clean(text):
    text = text.lower()
    text = re.sub(r'http\S+|@\w+|[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()
df['clean'] = df['text'].apply(clean)

## 3. Feature Extraction

In [1]:
vec = TfidfVectorizer(max_features=5000, stop_words='english')
X = vec.fit_transform(df['clean']).toarray()
y = df['airline_sentiment'].map({'negative':0,'neutral':1,'positive':2})
print('Features:', X.shape)

In [1]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape[0], 'Test:', X_test.shape[0])

Train: 11712, Test: 2928


## 4. Train

In [1]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print('Accuracy: %.4f' % accuracy_score(y_test, model.predict(X_test)))

Accuracy: 0.7542


## 5. Test

In [1]:
tests = ['Great flight!', 'Terrible delay!!', 'Okay flight']
for t in tests:
    v = vec.transform([clean(t)]).toarray()
    p = model.predict(v)[0]
    print(f"'{t}' -> {['Negative','Neutral','Positive'][p]}")

'Great flight!' -> Positive
'Terrible delay!!' -> Negative
'Okay flight' -> Neutral
